# Runtime check: is there time per step for a learned-model planner?

Runs `scripts/runtime_check.py` on Kaggle hardware. Per scenario it measures:
1. **Simulator cost:** a full 8,640-step rollout with the starter kit's `bang_bang` policy (the cheapest policy, so this is close to pure simulator time).
2. **Model cost:** the Phase 2C graph `paired` ensemble (3 seeds, the fold that holds the building out) predicting N candidate action sequences of horizon H from the rollout's live context. Featurization and the network are timed separately.
3. **Budget table:** per-step headroom under three hypotheses for the evaluation allowance (`docs/phase0_findings.md` §2.4), and whether each planner configuration fits:
   - 48 scenarios in 9 h (675 s each);
   - 48 scenarios in 12 h (900 s each);
   - the paper's 10× `bang_bang` per rollout.

Two scenarios by default: `building_train_8/atlanta_1` (smallest, fastest simulator: binding under the 10× rule) and `building_train_2/atlanta_3` (largest, slowest simulator: binding under a fixed allowance per scenario). Expected wall time: well under 1 h.

**Kaggle settings:** Accelerator **None (CPU)**, which is the conservative assumption for the graded run. Internet **on** (setup only).

**Inputs** (Add Input):
- the dataset with the Phase 1 sample (`phase2c_train.zip`, or `phase1_sample.zip`): the model features need its per-scenario `meta.json`;
- the Phase 2C checkpoints: the **output of `phase2c_train`** (latest version), or the resume dataset. Any `.../results/phase2c/training` folder with `graph_paired_lam1/` works.

**Output:** `/kaggle/working/runtime_check/runtime_check.json` (all timings + budget tables) and the printed tables below.

**Code:** cell 1 clones `BRANCH` from GitHub, so `scripts/runtime_check.py` must be **pushed** to that branch first.

In [ ]:
import os, glob, json, subprocess, time
BRANCH = "phase2c-3rd-run"        # must contain scripts/runtime_check.py (push it first)
STARTER_KIT_REV = "1fb75606d12e016463fbc63de704967c5b2bf9d5"
SCENARIOS = ["building_train_8/atlanta_1", "building_train_2/atlanta_3"]
STEPS = 8640                      # full episode; lower only for a quick smoke test
SIZES = [1, 20, 64, 128]          # candidate action sequences per model call
HORIZONS = [12, 36]
REPEATS = 5
THREADS = os.cpu_count()          # torch threads for the model timing

REPO = "/tmp/rt/smart-building-rl"; PY = "/tmp/rt/venv/bin/python"; KIT = f"{REPO}/starter-kit"
INPUT_ROOT = "/kaggle/input"; OUT = "/kaggle/working/runtime_check"
os.makedirs(OUT, exist_ok=True)
os.environ["MPLBACKEND"] = "Agg"
!lscpu | grep -E 'Model name|^CPU\(s\)|Thread|MHz' ; free -g | head -2

In [ ]:
# ---- environment (CPU torch) ----
!mkdir -p /tmp/rt && rm -rf {REPO} && git clone -q -b {BRANCH} https://github.com/fannam/smart-building-rl.git {REPO}
!pip install -q uv
!uv venv -q --python 3.12 /tmp/rt/venv
!git clone -q https://github.com/smart-buildings-2026/starter-kit.git {KIT} && cd {KIT} && git checkout -q {STARTER_KIT_REV}
!uv pip install -q -p {PY} -e {KIT}
!uv pip install -q -p {PY} "numpy==2.0.2" "pandas==2.3.3"
!uv pip install -q -p {PY} --index-url https://download.pytorch.org/whl/cpu torch
!cd {REPO} && git log --oneline -1 && test -f scripts/runtime_check.py && echo "runtime_check.py present"
!{PY} -c "import torch; print('torch', torch.__version__, 'threads', torch.get_num_threads())"

In [ ]:
# ---- inputs: Phase 1 sample (meta.json) + Phase 2C checkpoints ----
def find_dir(pattern, must_contain):
    for d in sorted(glob.glob(f"{INPUT_ROOT}/**/{pattern}", recursive=True)):
        if os.path.exists(os.path.join(d, must_contain)):
            return d
    return None

p1 = find_dir("phase1_sample", "manifest.jsonl")
if p1 is None:
    z = glob.glob(f"{INPUT_ROOT}/**/phase1_sample.zip", recursive=True) or glob.glob(f"{INPUT_ROOT}/**/phase2c_train.zip", recursive=True)
    assert z, "attach the dataset with the Phase 1 sample (phase2c_train.zip or phase1_sample.zip)"
    subprocess.run(["unzip", "-q", z[0], "-d", "/tmp/rt/data"], check=True)
    p1 = find_dir("phase1_sample", "manifest.jsonl") or next(d for d in glob.glob("/tmp/rt/data/**/phase1_sample", recursive=True)
                                                              if os.path.exists(os.path.join(d, "manifest.jsonl")))
runs = find_dir("results/phase2c/training", "graph_paired_lam1")
assert runs, "attach the phase2c_train output (or the resume dataset): need .../results/phase2c/training/graph_paired_lam1/"
print("phase1_sample:", p1)
print("checkpoints:  ", runs)
print("graph_paired_lam1 checkpoints:", len(glob.glob(f"{runs}/graph_paired_lam1/*/model_seed*.pt")))

In [ ]:
# ---- run (prints progress; full rollouts take several minutes each) ----
t0 = time.time()
cmd = (f"cd {REPO} && {PY} -u scripts/runtime_check.py --kit {KIT} --runs {runs} --phase1-root {p1} "
       f"--scenarios {' '.join(SCENARIOS)} --steps {STEPS} --sizes {' '.join(map(str, SIZES))} "
       f"--horizons {' '.join(map(str, HORIZONS))} --repeats {REPEATS} --threads {THREADS} --out {OUT} "
       f"2>&1 | grep --line-buffered -v -i warning")
subprocess.run(cmd, shell=True)
print(f"wall {(time.time() - t0) / 60:.1f} min")

In [ ]:
# ---- summary ----
R = json.load(open(f"{OUT}/runtime_check.json"))
print(json.dumps(R["machine"], indent=1))
for sc, r in R["scenarios"].items():
    s, b = r["simulator"], r["budget"]
    print(f"\n{sc}: K={s['K']} V={s['V']}  rollout {b['rollout_full_episode_s']:.0f}s  "
          f"env.step mean {s['env_step']['mean_ms']:.1f} ms  bang_bang get_action {s['get_action']['mean_ms']:.2f} ms")
    for k, h in b["headroom_ms_per_step"].items():
        print(f"  headroom [{k}]: {h:.1f} ms/step")
!cd /kaggle/working && zip -qr runtime_check.zip runtime_check && ls -lh runtime_check.zip